# S11 · leakdepth worked examples (NB1)

Runs the frozen design D9 (S11_STAGE4_DESIGN_v1): sensitivity curves on the CIC-IDS2017 and UNSW-NB15
Hugging Face mirrors (PRELIMINARY data), the RF duplicate-level diagnostic, and the lineage positive and negative
controls. Use a **High-RAM CPU** runtime. Every step writes to `MyDrive/S11-SecSplit/nb1/` and resumes after a
disconnect: rerun the setup cell, then the cell that was interrupted. Each cell ends with a compact summary.

In [ ]:
# 1. Setup: Drive, embedded leakdepth wheel (hash-checked), versions
import base64, hashlib, json, os, subprocess, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/S11-SecSplit/nb1"
os.makedirs(OUT, exist_ok=True)
WHEEL_SHA = "6194495b5dff78f7b98794e04870d535404c1c949d4cfa28a54469abd764737d"  # package files identical to the tested wheel 7da15ab81b61…; author metadata set to the release author
WHEEL_B64 = "..."
w = "/content/leakdepth-0.1.0-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "wheel hash mismatch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow", "huggingface_hub"], check=True)
import numpy as np, pandas as pd, sklearn, leakdepth as L
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| CPUs", os.cpu_count(), "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def curve_summary(rep):
    c = pd.DataFrame(rep.results["curve"])
    keep = [k for k in ["unit", "model", "F_mean", "d_raw_mean", "d_train_mean", "d_full_mean", "test_attack_share_mean", "n_seeds"] if k in c]
    with pd.option_context("display.width", 200, "display.precision", 4):
        print(c[keep].to_string(index=False))
    for i in rep.results["invalid_levels"]:
        print("INVALID", i["seed"], i["unit"], "; ".join(i["reasons"])[:160])


In [ ]:
# 2. Data: Hugging Face mirrors, SHA-256 checked against the sandbox manifest
from huggingface_hub import hf_hub_download
MANIFEST = [{"repo": "rdpahalavan/CIC-IDS2017", "file": "Network-Flows/CICIDS_Flow.parquet", "bytes": 369814810, "sha256": "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35"}, {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00000-of-00002.parquet", "bytes": 124419404, "sha256": "2aada2a26d061111f4e8fb84e716f5f11264fee71abe04697d42cb89e488d047"}, {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00001-of-00002.parquet", "bytes": 105596940, "sha256": "7c6699ae967567977dee9e9193543b515255f4e1671ca79bf9ae65e7866ffef1"}]
DATA = "/content/data"
PATHS = {}
for m in MANIFEST:
    p = hf_hub_download(m["repo"], m["file"], repo_type="dataset", local_dir=f"{DATA}/{m['repo'].replace('/', '__')}")
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    print(m["repo"], m["file"], "OK" if h == m["sha256"] else "MISMATCH " + h)
    assert h == m["sha256"], "mirror changed: results would not be comparable"
    PATHS[m["file"]] = p
CIC_PATH = PATHS["Network-Flows/CICIDS_Flow.parquet"]
UNSW_PATHS = [PATHS[f] for f in sorted(PATHS) if "UNSW" in f or f.startswith("data/train")]


In [ ]:
# 3. CIC-IDS2017 sensitivity curves (levels of design D9; models hgb, rf, lr; seeds 0-2; resumable)
LEVELS = ["dup", "5tuple", "hostpair", "session:60", "session:300", "session:1800", "capture"]
cic = L.load(pd.read_parquet(CIC_PATH), "cic-ids2017", name="CIC-IDS2017 (HF mirror)")
for model in ["hgb", "rf", "lr"]:
    t0 = time.time()
    rep = L.sensitivity(cic, LEVELS, models=[model], seeds=[0, 1, 2], resume=f"{OUT}/cic_{model}_folds.jsonl")
    rep.to_json(f"{OUT}/cic_{model}_sensitivity.json")
    print(f"== CIC {model} ({time.time() - t0:.0f} s)")
    curve_summary(rep)


In [ ]:
# 4. UNSW-NB15 sensitivity curves (profile transform blank_to_zero=True, as in pilot amendment A2)
unsw_df = pd.concat([pd.read_parquet(p) for p in UNSW_PATHS], ignore_index=True)
unsw = L.load(unsw_df, "unsw-nb15", name="UNSW-NB15 (HF mirror)", blank_to_zero=True)
for model in ["hgb", "rf", "lr"]:
    t0 = time.time()
    rep = L.sensitivity(unsw, LEVELS, models=[model], seeds=[0, 1, 2], resume=f"{OUT}/unsw_{model}_folds.jsonl")
    rep.to_json(f"{OUT}/unsw_{model}_sensitivity.json")
    print(f"== UNSW {model} ({time.time() - t0:.0f} s)")
    curve_summary(rep)


In [ ]:
# 5. RF duplicate-level diagnostic on CIC (seed 0): per-class recall, duplicate structure, RF settings
from leakdepth import groups as G
from leakdepth.split import constrained_group_split
from leakdepth.sensitivity import make_model, stratified_cap
from leakdepth.metrics import macro_f1
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
X, y, cls = cic.X("float32"), cic.y, cic.cls
g = G.extract(cic, "dup")
size = pd.Series(g).map(pd.Series(g).value_counts()).to_numpy()
tr0, te0, _ = constrained_group_split(np.arange(len(cic)), cls, y, 0.2, 0, "BENIGN")
trd, ted, vd = constrained_group_split(g, cls, y, 0.2, 0, "BENIGN")
rng = np.random.default_rng(10_000)
models = {"rf_default": lambda: make_model("rf", 0),
          "rf_no_bootstrap": lambda: make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(100, bootstrap=False, n_jobs=-1, random_state=0)),
          "rf_min_leaf5": lambda: make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(100, min_samples_leaf=5, n_jobs=-1, random_state=0)),
          "hgb": lambda: make_model("hgb", 0)}
rows, recall = [], {}
for split, (tr, te) in {"row": (tr0, te0), "dup": (trd, ted)}.items():
    trc = stratified_cap(tr, cls, 300_000, rng)
    print(f"{split}: test rows in duplicate clusters (size>1) {np.mean(size[te] > 1):.3f}; train {np.mean(size[trc] > 1):.3f}; test attack share {y[te].mean():.3f}")
    for name, mk in models.items():
        m = mk().fit(X.iloc[trc], y[trc]); p = m.predict(X.iloc[te])
        rows.append({"split": split, "model": name, "F": round(macro_f1(y[te], p), 4)})
        recall[(split, name)] = pd.Series(p == y[te]).groupby(cls[te]).mean()
print(pd.DataFrame(rows).pivot(index="model", columns="split", values="F").assign(drop=lambda d: d["row"] - d["dup"]).round(4).to_string())
rec = pd.DataFrame(recall).round(3)
print(rec[[c for c in rec.columns if c[1] in ("rf_default", "hgb")]].to_string())
json.dump({"F": rows, "per_class_accuracy": {f"{a}|{b}": v.to_dict() for (a, b), v in recall.items()}},
          open(f"{OUT}/cic_rf_dup_diagnostic.json", "w"), indent=1, default=float)


In [ ]:
# 6. Lineage controls. Positive: Ahuja vs IoT-DH v1 (row identity reported in AlQahtani, in preparation; software
# doi:10.5281/zenodo.23599584). Negatives: Ahuja vs Karmous / SNT / UKMDDoSFlow, CIC vs UNSW.
# Files: S06 raw copies on Drive; IoT-DH v1 from MyDrive/S11-SecSplit/raw/ (CSV or the Mendeley zip) or the Mendeley public API.
import glob, zipfile, urllib.request
RAW6 = "/content/drive/MyDrive/S06-OFDG/raw"
RAW11 = "/content/drive/MyDrive/S11-SecSplit/raw"
os.makedirs(RAW11, exist_ok=True)

def iotdh_v1():
    hits = glob.glob(f"{RAW11}/*IoT-DH*.csv") + glob.glob(f"{RAW11}/*IoT-DH*.zip")
    if not hits:
        try:
            api = "https://data.mendeley.com/public-api/datasets/8dns3xbckv/files?folder_id=root&version=1"
            for f in json.load(urllib.request.urlopen(api, timeout=60)):
                url = f["content_details"]["download_url"]
                dest = f"{RAW11}/{f['filename']}"
                print("downloading", f["filename"]); urllib.request.urlretrieve(url, dest); hits.append(dest)
        except Exception as e:
            raise SystemExit(f"IoT-DH v1 not found ({e}). Upload 'IoT-DH Dataset.zip' (v1, 728,853,142 bytes) to {RAW11} and rerun.")
    p = [h for h in hits if h.endswith(".csv")] or hits
    p = p[0]
    if p.endswith(".zip"):
        z = zipfile.ZipFile(p); name = [n for n in z.namelist() if n.endswith(".csv")][0]
        return pd.read_csv(z.open(name), low_memory=False)
    return pd.read_csv(p, low_memory=False)

ahuja = pd.read_csv(f"{RAW6}/ahuja__dataset_sdn.csv", low_memory=False)
assert len(ahuja) == 104_345, len(ahuja)
res = {}
def run(name, a, b, na, nb):
    if done(f"lineage_{name}.json"):
        d = json.load(open(f"{OUT}/lineage_{name}.json"))
    else:
        t0 = time.time(); r = L.lineage(a, b, na, nb); r.to_json(f"{OUT}/lineage_{name}.json"); d = r.to_dict()
        print(f"({time.time() - t0:.0f} s)")
    res[name] = [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 5), c["aligned_columns"]) for c in d["results"]["containment"]]
    print(name, res[name] or [f["message"] for f in d["findings"]])

for nm, f in [("karmous", "karmous__SDN_DDoS_Dataset.csv"), ("snt", "snt__SNT_dataset.csv"), ("ukm", "ukmddosflow__flow_features.csv")]:
    run(f"negative_ahuja_{nm}", ahuja, pd.read_csv(f"{RAW6}/{f}", low_memory=False), "Ahuja", nm)
run("negative_cic_unsw", pd.read_parquet(CIC_PATH), unsw_df, "CIC-IDS2017", "UNSW-NB15")
iot = iotdh_v1(); assert len(iot) == 15_817_209, len(iot)
run("positive_ahuja_iotdhv1", ahuja, iot, "Ahuja", "IoT-DH v1"); del iot


In [ ]:
# 7. Compact summary of everything in OUT (paste this output back)
summary = {}
for f in sorted(glob.glob(f"{OUT}/*_sensitivity.json")):
    d = json.load(open(f))
    summary[os.path.basename(f)] = [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()
                                     if k in ("unit", "model", "F_mean", "d_raw_mean", "d_train_mean", "d_full_mean", "d_full_std", "n_seeds")}
                                    for r in d["results"]["curve"]] + [{"invalid": i["unit"], "seed": i["seed"]} for i in d["results"]["invalid_levels"]]
for f in sorted(glob.glob(f"{OUT}/lineage_*.json")):
    d = json.load(open(f))
    summary[os.path.basename(f)] = [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 5)) for c in d["results"]["containment"]]
if done("cic_rf_dup_diagnostic.json"):
    summary["cic_rf_dup_diagnostic"] = json.load(open(f"{OUT}/cic_rf_dup_diagnostic.json"))["F"]
json.dump(summary, open(f"{OUT}/S11_NB1_summary.json", "w"), indent=1)
print(json.dumps(summary, separators=(",", ":")))
